## Types of WorkFlows in LangGraph   
1. Sequential Workflows in LangGraph
2. Parallel Workflows:
3. Conditional Workflows
4. Itrartive WorkFlows

### 1. Sequential Workflows in LangGraph

https://shahil04.medium.com/sequential-workflows-in-langgraph-agentic-ai-using-langgraph-class-5-251bed047cce

Already done in 1st class 
- BMI Calculator
- BASIC LLM WORKFLOW
-  Prompt Chaining Workflow (Blog Generator)


In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv
load_dotenv()
model = ChatOpenAI()
# create a state

class LLMState(TypedDict):
    question: str
    answer: str

def llm_qa(state: LLMState) -> LLMState:
    # extract the question from state
    question = state['question']
    # form a prompt
    prompt = f'Answer the following question {question}'
    # ask that question to the LLM
    answer = model.invoke(prompt).content
    # update the answer in the state
    state['answer'] = answer

    return state

# create our graph
graph = StateGraph(LLMState)
# add nodes
graph.add_node('llm_qa', llm_qa)
# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)
# compile
workflow = graph.compile()
# execute

intial_state = {'question': 'How far is moon from the earth?'}
final_state = workflow.invoke(intial_state)
print(final_state['answer'])


#  this is equivalent to langraph execution but directly invoking the model with the prompt 
model.invoke('How far is moon from the earth?').content
# but here we leern how to use langgraph to create a workflow and execute it. not just ask a question to the model.

### 2. Parallel Workflows:
1. Practical Example 1: Cricket Batsman Stats (Non-LLM Workflow)

- 1. The Goal and Workflow Structure

- Input: The system receives a batsman’s runs, balls, fours, and sixes.
- Parallel Nodes: The system simultaneously calculates:
1. Strike Rate: (Runs / Balls) * 100.
2. Balls Per Boundary (BPB): Balls / (Fours + Sixes).
3. Runs in Boundary Percentage: ((Fours * 4 + Sixes * 6) / Runs) * 100.


In [ ]:
# Import the required classes from LangGraph
# StateGraph -> used to create the workflow/graph
# START and END -> special nodes representing the beginning and end of the workflow
from langgraph.graph import StateGraph, START, END

# TypedDict is used to define the structure of our state
from typing import TypedDict


# Define the state structure for the batsman
# Every node in the graph can read values from this state
class BatsmanState(TypedDict):

    # Input statistics
    runs: int
    balls: int
    fours: int
    sixes: int

    # Calculated statistics
    sr: float                  # Strike rate
    bpb: float                 # Balls per boundary
    boundary_percent: float    # Percentage of runs scored through boundaries

    # Final textual summary
    summary: str


# Node 1: Calculate the batsman's strike rate
def calculate_sr(state: BatsmanState):

    # Strike Rate = (Runs / Balls) * 100
    sr = (state['runs'] / state['balls']) * 100

    # Return the calculated value
    # This value will be added to the graph state
    return {'sr': sr}


# Node 2: Calculate balls per boundary
def calculate_bpb(state: BatsmanState):

    # Total boundaries = fours + sixes
    # Balls per boundary = total balls / total boundaries
    bpb = state['balls'] / (state['fours'] + state['sixes'])

    # Return the calculated value
    return {'bpb': bpb}


# Node 3: Calculate the percentage of runs
# that came from fours and sixes
def calculate_boundary_percent(state: BatsmanState):

    # Runs scored through fours = fours * 4
    # Runs scored through sixes = sixes * 6
    # Boundary percentage = boundary runs / total runs * 100
    boundary_percent = (
        ((state['fours'] * 4) + (state['sixes'] * 6))
        / state['runs']
    ) * 100

    # Return the calculated value
    return {'boundary_percent': boundary_percent}


# Node 4: Create a human-readable summary
def summary(state: BatsmanState):

    # Create a formatted string using the calculated statistics
    summary = f"""
Strike Rate - {state['sr']}

Balls per boundary - {state['bpb']}

Boundary percent - {state['boundary_percent']}
"""

    # Return the summary
    return {'summary': summary}


# Create a StateGraph using BatsmanState
# This graph will manage the flow of data between our functions
graph = StateGraph(BatsmanState)


# --------------------------------------------------
# ADD NODES
# --------------------------------------------------

# Each function is added as a node in the graph
graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_boundary_percent', calculate_boundary_percent)
graph.add_node('summary', summary)


# --------------------------------------------------
# ADD EDGES
# --------------------------------------------------

# The three calculations can start directly from START
# They are independent of each other
graph.add_edge(START, 'calculate_sr')
graph.add_edge(START, 'calculate_bpb')
graph.add_edge(START, 'calculate_boundary_percent')


# Once all calculations are completed,
# their results are passed to the summary node
graph.add_edge('calculate_sr', 'summary')
graph.add_edge('calculate_bpb', 'summary')
graph.add_edge('calculate_boundary_percent', 'summary')


# After the summary is generated,
# the workflow reaches the END node
graph.add_edge('summary', END)


# Compile the graph into an executable workflow
workflow = graph.compile()


# Display the compiled workflow
workflow


# --------------------------------------------------
# INITIAL STATE
# --------------------------------------------------

# Input data for the batsman
initial_state = {
    'runs': 100,
    'balls': 50,
    'fours': 6,
    'sixes': 4
}


# --------------------------------------------------
# RUN THE WORKFLOW
# --------------------------------------------------

# Pass the initial state into the workflow
# LangGraph executes the connected nodes and
# returns the final state
workflow.invoke(initial_state)

### Practical Example 2: UPSC Essay Evaluator (LLM-Based Workflow) Parallel workflow

1. The Goal and Workflow Structure

- Input: An essay text submitted by a user.
- Parallel Nodes: The essay is sent to three different LLMs simultaneously to evaluate three distinct aspects:
1. Clarity of Thought.
2. Depth of Analysis.
3. Language Quality.

- Output Requirement: Each parallel LLM must return two specific things: a textual feedback string and an integer score between 0 and 10.
- Final Evaluation Node: This node gathers the three text feedbacks and uses another LLM to merge them into a summarised_feedback. It also calculates the mathematical average_score of the three numerical scores

In [ ]:
# Import LangGraph components
# StateGraph -> used to create the workflow
# START -> starting point of the workflow
# END -> ending point of the workflow
from langgraph.graph import StateGraph, START, END

# ChatOpenAI is used to interact with an OpenAI chat model
from langchain_openai import ChatOpenAI

# Used to load environment variables such as OPENAI_API_KEY
from dotenv import load_dotenv

# TypedDict is used to define the structure of our LangGraph state
# Annotated is used to specify how a particular state field should be updated
from typing import TypedDict, Annotated

# Pydantic classes are used to define structured output
from pydantic import BaseModel, Field

# operator.add will be used to combine scores coming from
# multiple parallel nodes
import operator


# Load environment variables from the .env file
load_dotenv()


# Create the LLM
model = ChatOpenAI(model='gpt-4o-mini')


# --------------------------------------------------
# STRUCTURED OUTPUT SCHEMA
# --------------------------------------------------

# This Pydantic model defines the format in which
# we want the LLM to return its evaluation
class EvaluationSchema(BaseModel):

    # Detailed feedback about the essay
    feedback: str = Field(
        description='Detailed feedback for the essay'
    )

    # Score between 0 and 10
    # ge=0 means minimum value is 0
    # le=10 means maximum value is 10
    score: int = Field(
        description='Score out of 10',
        ge=0,
        le=10
    )


# Convert the normal LLM into a structured-output LLM
# Instead of returning arbitrary text, the model will
# return data matching EvaluationSchema
structured_model = model.with_structured_output(EvaluationSchema)


# --------------------------------------------------
# FIRST EXAMPLE ESSAY
# --------------------------------------------------

# Sample essay that we want to evaluate
essay = """India in the Age of AI
As the world enters a transformative era defined by artificial intelligence (AI), India stands at a critical juncture — one where it can either emerge as a global leader in AI innovation or risk falling behind in the technology race. The age of AI brings with it immense promise as well as unprecedented challenges, and how India navigates this landscape will shape its socio-economic and geopolitical future.

India's strengths in the AI domain are rooted in its vast pool of skilled engineers, a thriving IT industry, and a growing startup ecosystem. With over 5 million STEM graduates annually and a burgeoning base of AI researchers, India possesses the intellectual capital required to build cutting-edge AI systems. Institutions like IITs, IIITs, and IISc have begun fostering AI research, while private players such as TCS, Infosys, and Wipro are integrating AI into their global services. In 2020, the government launched the National AI Strategy (AI for All) with a focus on inclusive growth, aiming to leverage AI in healthcare, agriculture, education, and smart mobility.

One of the most promising applications of AI in India lies in agriculture, where predictive analytics can guide farmers on optimal sowing times, weather forecasts, and pest control. In healthcare, AI-powered diagnostics can help address India’s doctor-patient ratio crisis, particularly in rural areas. Educational platforms are increasingly using AI to personalize learning paths, while smart governance tools are helping improve public service delivery and fraud detection.

However, the path to AI-led growth is riddled with challenges. Chief among them is the digital divide. While metropolitan cities may embrace AI-driven solutions, rural India continues to struggle with basic internet access and digital literacy. The risk of job displacement due to automation also looms large, especially for low-skilled workers. Without effective skilling and re-skilling programs, AI could exacerbate existing socio-economic inequalities.

Another pressing concern is data privacy and ethics. As AI systems rely heavily on vast datasets, ensuring that personal data is used transparently and responsibly becomes vital. India is still shaping its data protection laws, and in the absence of a strong regulatory framework, AI systems may risk misuse or bias.

To harness AI responsibly, India must adopt a multi-stakeholder approach involving the government, academia, industry, and civil society. Policies should promote open datasets, encourage responsible innovation, and ensure ethical AI practices. There is also a need for international collaboration, particularly with countries leading in AI research, to gain strategic advantage and ensure interoperability in global systems.

India’s demographic dividend, when paired with responsible AI adoption, can unlock massive economic growth, improve governance, and uplift marginalized communities. But this vision will only materialize if AI is seen not merely as a tool for automation, but as an enabler of human-centered development.

In conclusion, India in the age of AI is a story in the making — one of opportunity, responsibility, and transformation. The decisions we make today will not just determine India’s AI trajectory, but also its future as an inclusive, equitable, and innovation-driven society."""


# Create a prompt asking the model to evaluate the essay
prompt = f'''
Evaluate the language quality of the following essay
and provide feedback and assign a score out of 10.

{essay}
'''


# Directly invoke the structured model
# The returned object will contain:
# output.feedback
# output.score
structured_model.invoke(prompt).feedback


# ==================================================
# LANGGRAPH STATE
# ==================================================

# This defines the complete state that will move
# through our LangGraph workflow
class UPSCState(TypedDict):

    # Original essay
    essay: str

    # Feedback generated by individual evaluators
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str

    # Final combined feedback
    overall_feedback: str

    # Scores generated by individual evaluators
    #
    # Annotated[list[int], operator.add] tells LangGraph
    # how to merge values when multiple nodes update
    # this field.
    #
    # Example:
    # [7] + [6] + [8] = [7, 6, 8]
    individual_scores: Annotated[list[int], operator.add]

    # Average of all individual scores
    avg_score: float


# ==================================================
# NODE 1: LANGUAGE EVALUATION
# ==================================================

def evaluate_language(state: UPSCState):

    # Create a prompt specifically asking the LLM
    # to evaluate the language quality
    prompt = f'''
    Evaluate the language quality of the following essay
    and provide feedback and assign a score out of 10.

    {state["essay"]}
    '''

    # Ask the structured model to evaluate the essay
    output = structured_model.invoke(prompt)

    # Return the language feedback and score
    return {
        'language_feedback': output.feedback,
        'individual_scores': [output.score]
    }


# ==================================================
# NODE 2: ANALYSIS EVALUATION
# ==================================================

def evaluate_analysis(state: UPSCState):

    # Ask the model to evaluate the depth of analysis
    prompt = f'''
    Evaluate the depth of analysis of the following essay
    and provide feedback and assign a score out of 10.

    {state["essay"]}
    '''

    # Get structured response
    output = structured_model.invoke(prompt)

    # Store feedback and score in the state
    return {
        'analysis_feedback': output.feedback,
        'individual_scores': [output.score]
    }


# ==================================================
# NODE 3: CLARITY OF THOUGHT EVALUATION
# ==================================================

def evaluate_thought(state: UPSCState):

    # Ask the model to evaluate how clearly
    # the ideas are expressed
    prompt = f'''
    Evaluate the clarity of thought of the following essay
    and provide feedback and assign a score out of 10.

    {state["essay"]}
    '''

    # Get structured response
    output = structured_model.invoke(prompt)

    # Return the feedback and score
    return {
        'clarity_feedback': output.feedback,
        'individual_scores': [output.score]
    }


# ==================================================
# NODE 4: FINAL EVALUATION
# ==================================================

def final_evaluation(state: UPSCState):

    # --------------------------------------------------
    # GENERATE OVERALL FEEDBACK
    # --------------------------------------------------

    # Combine all three evaluator feedbacks
    # and ask the LLM to create one final summary
    prompt = f'''
    Based on the following feedbacks, create a summarized feedback.

    Language feedback:
    {state["language_feedback"]}

    Depth of analysis feedback:
    {state["analysis_feedback"]}

    Clarity of thought feedback:
    {state["clarity_feedback"]}
    '''

    # Use the normal model because we only need text here,
    # not structured JSON/Pydantic output
    overall_feedback = model.invoke(prompt).content


    # --------------------------------------------------
    # CALCULATE AVERAGE SCORE
    # --------------------------------------------------

    # individual_scores contains the scores from
    # all three evaluation nodes.
    #
    # Example:
    # [8, 6, 7]
    #
    # Average:
    # (8 + 6 + 7) / 3 = 7.0
    avg_score = (
        sum(state['individual_scores'])
        / len(state['individual_scores'])
    )


    # Return the final feedback and average score
    return {
        'overall_feedback': overall_feedback,
        'avg_score': avg_score
    }


# ==================================================
# CREATE THE GRAPH
# ==================================================

# Create a StateGraph using UPSCState
graph = StateGraph(UPSCState)


# --------------------------------------------------
# ADD NODES
# --------------------------------------------------

# Each function becomes a node in the LangGraph
graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)


# ==================================================
# ADD EDGES
# ==================================================

# All three evaluation nodes start independently
# from START.
#
# This means they can execute in parallel.
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')


# All three evaluation nodes send their results
# to the final evaluation node
graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')


# After final evaluation, the workflow ends
graph.add_edge('final_evaluation', END)


# ==================================================
# COMPILE THE GRAPH
# ==================================================

# Compile the graph into an executable workflow
workflow = graph.compile()


# Display the compiled workflow
workflow


# ==================================================
# SECOND ESSAY
# ==================================================

# This is a deliberately weaker essay.
# We will pass it through our evaluation workflow.
essay2 = """India and AI Time

Now world change very fast because new tech call Artificial Intel… something (AI). India also want become big in this AI thing. If work hard, India can go top. But if no careful, India go back.

India have many good. We have smart student, many engine-ear, and good IT peoples. Big company like TCS, Infosys, Wipro already use AI. Government also do program “AI for All”. It want AI in farm, doctor place, school and transport.

In farm, AI help farmer know when to put seed, when rain come, how stop bug. In health, AI help doctor see sick early. In school, AI help student learn good. Government office use AI to find bad people and work fast.

But problem come also. First is many villager no have phone or internet. So AI not help them. Second, many people lose job because AI and machine do work. Poor people get more bad.

One more big problem is privacy. AI need big big data. Who take care? India still make data rule. If no strong rule, AI do bad.

India must all people together – govern, school, company and normal people. We teach AI and make sure AI not bad. Also talk to other country and learn from them.

If India use AI good way, we become strong, help poor and make better life. But if only rich use AI, and poor no get, then big bad thing happen.

So, in short, AI time in India have many hope and many danger. We must go right road. AI must help all people, not only some. Then India grow big and world say "good job India"."""


# ==================================================
# INITIAL STATE
# ==================================================

# Put the essay into the initial LangGraph state
initial_state = {
    'essay': essay2
}


# ==================================================
# RUN THE WORKFLOW
# ==================================================

# Send the initial state into the graph
# The three evaluation nodes will run,
# their scores will be combined,
# and finally the overall feedback and average
# score will be generated.
workflow.invoke(initial_state)

# 3. Conditional Workflows


## Create a quadratic_equation_workflow using LangGraph

```
                 discriminant
                      │
                      ▼
                check_condition
                /      |       \
               /       |        \
             > 0      == 0       < 0
              │         │          │
              ▼         ▼          ▼
          real_roots  repeated   no_real_roots
```

In [ ]:
# Import the main LangGraph components
# StateGraph -> used to create the workflow
# START -> starting point of the graph
# END -> ending point of the graph
from langgraph.graph import StateGraph, START, END

# TypedDict -> used to define the structure of our state
# Literal -> used to specify the exact possible values
# that a function can return
from typing import TypedDict, Literal


# ==================================================
# STATE DEFINITION
# ==================================================

# Define the state of our quadratic-equation workflow
class QuadState(TypedDict):

    # Input values for the quadratic equation:
    # ax² + bx + c = 0
    a: int
    b: int
    c: int

    # Values that will be generated by the workflow
    equation: str
    discriminant: float
    result: str


# ==================================================
# NODE 1: SHOW EQUATION
# ==================================================

def show_equation(state: QuadState):

    # Create a string representation of the equation
    #
    # Example:
    # a = 2, b = 4, c = 2
    #
    # Output:
    # 2x2+4x2
    equation = f'{state["a"]}x2{state["b"]}x{state["c"]}'

    # Return the equation and update the state
    return {'equation': equation}


# ==================================================
# NODE 2: CALCULATE DISCRIMINANT
# ==================================================

def calculate_discriminant(state: QuadState):

    # Formula for the discriminant:
    #
    # D = b² - 4ac
    #
    # The discriminant tells us what type of roots
    # the quadratic equation has.
    discriminant = state["b"]**2 - (
        4 * state["a"] * state["c"]
    )

    # Store the discriminant in the state
    return {'discriminant': discriminant}


# ==================================================
# NODE 3: CALCULATE REAL ROOTS
# ==================================================

def real_roots(state: QuadState):

    # Quadratic formula:
    #
    # x = (-b ± √D) / 2a
    #
    # This node is executed only when:
    # discriminant > 0
    root1 = (
        -state["b"] + state["discriminant"]**0.5
    ) / (2 * state["a"])

    root2 = (
        -state["b"] - state["discriminant"]**0.5
    ) / (2 * state["a"])

    # Create a readable result
    result = f'The roots are {root1} and {root2}'

    # Return the result
    return {'result': result}


# ==================================================
# NODE 4: REPEATED ROOT
# ==================================================

def repeated_roots(state: QuadState):

    # When discriminant = 0, both roots are the same.
    #
    # Formula:
    # x = -b / 2a
    root = (
        -state["b"]
    ) / (2 * state["a"])

    # Create a readable result
    result = f'Only repeating root is {root}'

    # Return the result
    return {'result': result}


# ==================================================
# NODE 5: NO REAL ROOTS
# ==================================================

def no_real_roots(state: QuadState):

    # When the discriminant is less than zero,
    # the quadratic equation has no real roots.
    result = 'No real roots'

    # Return the result
    return {'result': result}


# ==================================================
# CONDITIONAL ROUTING FUNCTION
# ==================================================

def check_condition(
    state: QuadState
) -> Literal[
    "real_roots",
    "repeated_roots",
    "no_real_roots"
]:

    # Check the value of the discriminant.

    # If D > 0:
    # There are two different real roots.
    if state['discriminant'] > 0:
        return "real_roots"

    # If D = 0:
    # There is one repeated real root.
    elif state['discriminant'] == 0:
        return "repeated_roots"

    # If D < 0:
    # There are no real roots.
    else:
        return "no_real_roots"


# ==================================================
# CREATE GRAPH
# ==================================================

# Create a StateGraph using our QuadState
graph = StateGraph(QuadState)


# ==================================================
# ADD NODES
# ==================================================

# Add each function as a node in the graph
graph.add_node('show_equation', show_equation)
graph.add_node('calculate_discriminant', calculate_discriminant)
graph.add_node('real_roots', real_roots)
graph.add_node('repeated_roots', repeated_roots)
graph.add_node('no_real_roots', no_real_roots)


# ==================================================
# ADD NORMAL EDGES
# ==================================================

# Workflow starts by displaying the equation
graph.add_edge(START, 'show_equation')

# After displaying the equation,
# calculate the discriminant
graph.add_edge(
    'show_equation',
    'calculate_discriminant'
)


# ==================================================
# ADD CONDITIONAL EDGE
# ==================================================

# After calculating the discriminant,
# call check_condition().
#
# check_condition() decides which node should
# execute next.
graph.add_conditional_edges(
    'calculate_discriminant',
    check_condition
)


# ==================================================
# CONNECT THE THREE POSSIBLE END NODES
# ==================================================

# If discriminant > 0
graph.add_edge('real_roots', END)

# If discriminant = 0
graph.add_edge('repeated_roots', END)

# If discriminant < 0
graph.add_edge('no_real_roots', END)


# ==================================================
# COMPILE THE GRAPH
# ==================================================

# Compile the graph into an executable workflow
workflow = graph.compile()


# Display the compiled workflow
workflow


# ==================================================
# INITIAL STATE
# ==================================================

# Define the coefficients of:
#
# 2x² + 4x + 2 = 0
initial_state = {
    'a': 2,
    'b': 4,
    'c': 2
}


# ==================================================
# RUN WORKFLOW
# ==================================================

# Pass the initial state to the graph
workflow.invoke(initial_state)

## Create a AI Agent to review reply workflow using +ve ,or -ve review and based on review answer

In [ ]:
# Import LangGraph components
# StateGraph -> used to create the workflow
# START -> starting point of the graph
# END -> ending point of the graph
from langgraph.graph import StateGraph, START, END

# Import ChatOpenAI to interact with the OpenAI model
from langchain_openai import ChatOpenAI

# TypedDict -> used to define the structure of the graph state
# Literal -> restricts values to predefined options
from typing import TypedDict, Literal

# Used to load environment variables such as OPENAI_API_KEY
from dotenv import load_dotenv

# Pydantic is used to define structured output schemas
from pydantic import BaseModel, Field


# ==================================================
# LOAD ENVIRONMENT VARIABLES
# ==================================================

load_dotenv()


# ==================================================
# CREATE LLM
# ==================================================

# Create the OpenAI chat model
model = ChatOpenAI(model='gpt-4o-mini')


# ==================================================
# SENTIMENT SCHEMA
# ==================================================

# This schema defines the structure expected
# when the model determines the sentiment
class SentimentSchema(BaseModel):

    # The model can only return:
    # positive OR negative
    sentiment: Literal["positive", "negative"] = Field(
        description='Sentiment of the review'
    )


# ==================================================
# DIAGNOSIS SCHEMA
# ==================================================

# This schema is used to analyze negative reviews
class DiagnosisSchema(BaseModel):

    # Type of issue mentioned by the customer
    issue_type: Literal[
        "UX",
        "Performance",
        "Bug",
        "Support",
        "Other"
    ] = Field(
        description='The category of issue mentioned in the review'
    )

    # Emotional tone of the customer
    tone: Literal[
        "angry",
        "frustrated",
        "disappointed",
        "calm"
    ] = Field(
        description='The emotional tone expressed by the user'
    )

    # How urgent the issue appears to be
    urgency: Literal[
        "low",
        "medium",
        "high"
    ] = Field(
        description='How urgent or critical the issue appears to be'
    )


# ==================================================
# STRUCTURED MODELS
# ==================================================

# Create a structured model that returns
# SentimentSchema output
structured_model = model.with_structured_output(
    SentimentSchema
)


# Create another structured model that returns
# DiagnosisSchema output
structured_model2 = model.with_structured_output(
    DiagnosisSchema
)


# ==================================================
# TEST SENTIMENT MODEL
# ==================================================

# Test the sentiment classifier independently
prompt = 'What is the sentiment of the following review - The software too good'

# The result will be "positive"
structured_model.invoke(prompt).sentiment


# ==================================================
# LANGGRAPH STATE
# ==================================================

# This defines all the information that moves
# through the LangGraph workflow
class ReviewState(TypedDict):

    # Original customer review
    review: str

    # Sentiment detected by the model
    sentiment: Literal["positive", "negative"]

    # Diagnosis information for negative reviews
    diagnosis: dict

    # Final response sent to the customer
    response: str


# ==================================================
# NODE 1: FIND SENTIMENT
# ==================================================

def find_sentiment(state: ReviewState):

    # Create a prompt asking the model
    # to classify the review
    prompt = f'''
    For the following review find out the sentiment:

    {state["review"]}
    '''

    # Invoke the structured model
    # and extract the sentiment
    sentiment = structured_model.invoke(
        prompt
    ).sentiment

    # Update the state with the sentiment
    return {
        'sentiment': sentiment
    }


# ==================================================
# ROUTING FUNCTION
# ==================================================

def check_sentiment(
    state: ReviewState
) -> Literal[
    "positive_response",
    "run_diagnosis"
]:

    # If the review is positive,
    # directly generate a thank-you response
    if state['sentiment'] == 'positive':
        return 'positive_response'

    # Otherwise, send the review for diagnosis
    else:
        return 'run_diagnosis'


# ==================================================
# NODE 2: POSITIVE RESPONSE
# ==================================================

def positive_response(state: ReviewState):

    # Ask the LLM to generate a friendly response
    # for a positive review
    prompt = f"""
    Write a warm thank-you message in response
    to this review:

    "{state['review']}"

    Also, kindly ask the user to leave feedback
    on our website.
    """

    # Generate the response
    response = model.invoke(prompt).content

    # Store it in the state
    return {
        'response': response
    }


# ==================================================
# NODE 3: DIAGNOSE NEGATIVE REVIEW
# ==================================================

def run_diagnosis(state: ReviewState):

    # Ask the structured model to analyze
    # the negative review
    prompt = f"""
    Diagnose this negative review:

    {state['review']}

    Return issue_type, tone, and urgency.
    """

    # Invoke the diagnosis model
    response = structured_model2.invoke(prompt)

    # Convert the Pydantic object into a dictionary
    return {
        'diagnosis': response.model_dump()
    }


# ==================================================
# NODE 4: GENERATE NEGATIVE RESPONSE
# ==================================================

def negative_response(state: ReviewState):

    # Get the diagnosis generated by the previous node
    diagnosis = state['diagnosis']

    # Use the diagnosis to customize
    # the customer-support response
    prompt = f"""
    You are a support assistant.

    The user had a '{diagnosis['issue_type']}' issue,
    sounded '{diagnosis['tone']}', and marked urgency
    as '{diagnosis['urgency']}'.

    Write an empathetic, helpful resolution message.
    """

    # Generate the response
    response = model.invoke(prompt).content

    # Store the response in the state
    return {
        'response': response
    }


# ==================================================
# CREATE GRAPH
# ==================================================

# Create a StateGraph using ReviewState
graph = StateGraph(ReviewState)


# ==================================================
# ADD NODES
# ==================================================

graph.add_node(
    'find_sentiment',
    find_sentiment
)

graph.add_node(
    'positive_response',
    positive_response
)

graph.add_node(
    'run_diagnosis',
    run_diagnosis
)

graph.add_node(
    'negative_response',
    negative_response
)


# ==================================================
# ADD EDGES
# ==================================================

# Every review starts with sentiment analysis
graph.add_edge(
    START,
    'find_sentiment'
)


# After sentiment analysis,
# check_sentiment decides which path to take
graph.add_conditional_edges(
    'find_sentiment',
    check_sentiment
)


# Positive reviews go directly to END
graph.add_edge(
    'positive_response',
    END
)


# Negative reviews follow this path:
#
# run_diagnosis
#       ↓
# negative_response
#       ↓
#      END

graph.add_edge(
    'run_diagnosis',
    'negative_response'
)

graph.add_edge(
    'negative_response',
    END
)


# ==================================================
# COMPILE WORKFLOW
# ==================================================

# Compile the graph so it can be executed
workflow = graph.compile()


# Display the workflow
workflow


# ==================================================
# INITIAL STATE
# ==================================================

# Customer's negative review
initial_state = {
    'review': """
    I’ve been trying to log in for over an hour now,
    and the app keeps freezing on the authentication screen.
    I even tried reinstalling it, but no luck.

    This kind of bug is unacceptable,
    especially when it affects basic functionality.
    """
}


# ==================================================
# RUN WORKFLOW
# ==================================================

# Pass the review into the LangGraph workflow
workflow.invoke(initial_state)

# 4. Itrartive WorkFlows

# create a X_post_generaator 

The main idea

This is a generate → evaluate → improve loop:
```
                 ┌──────────────────────┐
                 │                      │
                 ▼                      │
START → GENERATE → EVALUATE ────────────┤
                     │                  │
                     │ approved         │
                     ▼                  │
                    END                │
                                        │
                     │ needs improvement
                     ▼                  │
                  OPTIMIZE ─────────────┘
```

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

# StateGraph -> used to create the LangGraph workflow
# START -> starting point of the graph
# END -> ending point of the graph
from langgraph.graph import StateGraph, START, END

# TypedDict -> defines the structure of our graph state
# Literal -> restricts a value to specific options
# Annotated -> allows us to define how state updates should be combined
from typing import TypedDict, Literal, Annotated

# OpenAI chat model
from langchain_openai import ChatOpenAI

# Message classes used to create structured prompts
from langchain_core.messages import SystemMessage, HumanMessage

# operator.add will be used to combine lists
# when multiple iterations update the same state field
import operator

# Pydantic is used for structured model output
from pydantic import BaseModel, Field


# ============================================================
# CREATE THE LLMs
# ============================================================

# LLM responsible for generating the initial tweet
generator_llm = ChatOpenAI(model='gpt-4o-mini')

# LLM responsible for evaluating the tweet
evaluator_llm = ChatOpenAI(model='gpt-4o-mini')

# LLM responsible for improving the tweet
optimizer_llm = ChatOpenAI(model='gpt-4o-mini')


# ============================================================
# EVALUATION SCHEMA
# ============================================================

# This schema defines exactly what the evaluator should return
class TweetEvaluation(BaseModel):

    # Evaluator can only return one of these two values
    evaluation: Literal[
        "approved",
        "needs_improvement"
    ] = Field(
        ...,
        description="Final evaluation result."
    )

    # Explanation of why the tweet was approved
    # or why it needs improvement
    feedback: str = Field(
        ...,
        description="Feedback for the tweet."
    )


# Convert the evaluator LLM into a structured-output model
#
# Instead of returning arbitrary text, the model must
# follow TweetEvaluation
structured_evaluator_llm = (
    evaluator_llm.with_structured_output(
        TweetEvaluation
    )
)


# ============================================================
# LANGGRAPH STATE
# ============================================================

class TweetState(TypedDict):

    # Topic provided by the user
    topic: str

    # Current version of the tweet
    tweet: str

    # Result of the evaluation
    evaluation: Literal[
        "approved",
        "needs_improvement"
    ]

    # Feedback generated by the evaluator
    feedback: str

    # Current iteration number
    iteration: int

    # Maximum number of optimization attempts
    max_iteration: int

    # Store every generated tweet
    #
    # operator.add means:
    #
    # [tweet1] + [tweet2] + [tweet3]
    #
    # becomes:
    #
    # [tweet1, tweet2, tweet3]
    tweet_history: Annotated[
        list[str],
        operator.add
    ]

    # Store feedback from every evaluation
    feedback_history: Annotated[
        list[str],
        operator.add
    ]


# ============================================================
# NODE 1: GENERATE TWEET
# ============================================================

def generate_tweet(state: TweetState):

    # Create messages for the generator LLM
    messages = [

        # System message defines the role of the model
        SystemMessage(
            content=(
                "You are a funny and clever "
                "Twitter/X influencer."
            )
        ),

        # Human message contains the actual task
        HumanMessage(
            content=f"""
Write a short, original, and hilarious tweet
on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm,
  or cultural references.
- Think in meme logic, punchlines,
  or relatable takes.
- Use simple, day to day English.
"""
        )
    ]

    # Generate the tweet
    response = generator_llm.invoke(messages).content

    # Store the generated tweet in the state
    #
    # tweet_history stores this tweet so we can
    # see all versions later
    return {
        'tweet': response,
        'tweet_history': [response]
    }


# ============================================================
# NODE 2: EVALUATE TWEET
# ============================================================

def evaluate_tweet(state: TweetState):

    # Create the evaluation prompt
    messages = [

        # Tell the model to act as a strict critic
        SystemMessage(
            content=(
                "You are a ruthless, no-laugh-given "
                "Twitter critic. You evaluate tweets "
                "based on humor, originality, virality, "
                "and tweet format."
            )
        ),

        # Provide the tweet and evaluation criteria
        HumanMessage(
            content=f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below:

1. Originality – Is this fresh, or have you seen
   it a hundred times before?

2. Humor – Did it genuinely make you smile,
   laugh, or chuckle?

3. Punchiness – Is it short, sharp,
   and scroll-stopping?

4. Virality Potential – Would people retweet
   or share it?

5. Format – Is it a well-formed tweet,
   not a setup-punchline joke, not a Q&A joke,
   and under 280 characters?

Auto-reject if:

- It is written in question-answer format
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- It ends with generic or weak lines

Respond ONLY in structured format:

- evaluation: "approved" or "needs_improvement"
- feedback: One paragraph explaining
  the strengths and weaknesses
"""
        )
    ]

    # Ask the structured evaluator to evaluate the tweet
    response = structured_evaluator_llm.invoke(messages)

    # Store evaluation, feedback and feedback history
    return {
        'evaluation': response.evaluation,
        'feedback': response.feedback,
        'feedback_history': [response.feedback]
    }


# ============================================================
# NODE 3: OPTIMIZE TWEET
# ============================================================

def optimize_tweet(state: TweetState):

    # Create a prompt containing:
    # - previous feedback
    # - topic
    # - current tweet
    messages = [

        # Define the role of the optimizer
        SystemMessage(
            content=(
                "You punch up tweets for virality "
                "and humor based on given feedback."
            )
        ),

        HumanMessage(
            content=f"""
Improve the tweet based on this feedback:

"{state['feedback']}"

Topic:
"{state['topic']}"

Original Tweet:
{state['tweet']}

Rewrite it as a short,
viral-worthy tweet.

Avoid Q&A style
and stay under 280 characters.
"""
        )
    ]

    # Generate an improved version
    response = optimizer_llm.invoke(messages).content

    # Increase iteration count
    iteration = state['iteration'] + 1

    # Update the current tweet and save it
    # into tweet_history
    return {
        'tweet': response,
        'iteration': iteration,
        'tweet_history': [response]
    }


# ============================================================
# ROUTING FUNCTION
# ============================================================

def route_evaluation(state: TweetState):

    # Stop the workflow if:
    #
    # 1. Tweet has been approved
    #
    # OR
    #
    # 2. Maximum number of iterations has been reached
    if (
        state['evaluation'] == 'approved'
        or
        state['iteration'] >= state['max_iteration']
    ):

        # Send the graph to END
        return 'approved'

    # Otherwise, send the tweet to optimizer
    else:
        return 'needs_improvement'


# ============================================================
# CREATE GRAPH
# ============================================================

graph = StateGraph(TweetState)


# ============================================================
# ADD NODES
# ============================================================

graph.add_node(
    'generate',
    generate_tweet
)

graph.add_node(
    'evaluate',
    evaluate_tweet
)

graph.add_node(
    'optimize',
    optimize_tweet
)


# ============================================================
# ADD EDGES
# ============================================================

# Workflow starts with tweet generation
graph.add_edge(
    START,
    'generate'
)

# Generated tweet goes to evaluator
graph.add_edge(
    'generate',
    'evaluate'
)


# ============================================================
# CONDITIONAL EDGE
# ============================================================

# After evaluation, route_evaluation()
# decides what happens next.
#
# approved -> END
#
# needs_improvement -> optimize
graph.add_conditional_edges(
    'evaluate',
    route_evaluation,
    {
        'approved': END,
        'needs_improvement': 'optimize'
    }
)


# After optimization, evaluate the new tweet again
#
# This creates a feedback loop:
#
# evaluate -> optimize -> evaluate -> optimize ...
graph.add_edge(
    'optimize',
    'evaluate'
)


# ============================================================
# COMPILE WORKFLOW
# ============================================================

# Compile the graph into an executable workflow
workflow = graph.compile()


# Display the compiled workflow
workflow


# ============================================================
# INITIAL STATE
# ============================================================

initial_state = {
    # Topic for the tweet
    "topic": "srhberhb",

    # Start from iteration 1
    "iteration": 1,

    # Maximum number of iterations
    "max_iteration": 5
}


# ============================================================
# RUN WORKFLOW
# ============================================================

result = workflow.invoke(initial_state)


# ============================================================
# DISPLAY RESULT
# ============================================================

result


# ============================================================
# DISPLAY ALL GENERATED TWEETS
# ============================================================

# tweet_history contains every version generated
# during the workflow
for tweet in result['tweet_history']:
    print(tweet)